# Kannolo Sparse Plain Index Benchmark (DotProduct)

## Import Required Libraries

In [1]:
from kannolo import SparseFlatIndex, SparsePlainHNSW
import numpy as np
import time
from pathlib import Path

## Load and Prepare Sparse Dataset

In [2]:
# Load sparse dataset from file (seismic format)
data_path = "/data1/knn_datasets/sparse_datasets/beir/beir_spladev3_fromCarlos_CROSSCHECK-MRR-BEFORE-USE/beir/quora/data/documents.bin"
print(f"Loading sparse data from: {data_path}")

# Load queries from file
queries_path = "/data3/silvio/quora_sparse_npy/queries/"
print(f"Loading sparse queries from: {queries_path}")

Loading sparse data from: /data1/knn_datasets/sparse_datasets/beir/beir_spladev3_fromCarlos_CROSSCHECK-MRR-BEFORE-USE/beir/quora/data/documents.bin
Loading sparse queries from: /data3/silvio/quora_sparse_npy/queries/


## Create Flat Index and Compute Ground Truth

In [3]:
# Create flat index from file for maximum dot product search
flat_index = SparseFlatIndex.build_from_file(data_path)
print("Sparse flat index built")

Sparse flat index built


In [4]:
# Load queries and compute ground truth (k=10)
k = 10

# Read queries from file assuming three flat .npy files: 
# - components.npy: componenents of the sparse query vectors concatenated, data type int32
# values.npy: values of the non-zero components of the sparse query vectors concatenated, data type float32
# offsets.npy: offsets of the start of each query in the components and values arrays, data type int64
# If your queries are in binary seismic format (.bin), use scripts/convert_bin_to_npy_arrays.py to convert them to the expected .npy format 
components = np.load(Path(queries_path) / "components.npy")
values = np.load(Path(queries_path) / "values.npy")
offsets = np.load(Path(queries_path) / "offsets.npy")
print(f"Loaded {len(offsets)-1} queries from .npy files")
n_queries = len(offsets) - 1

Loaded 10000 queries from .npy files


In [5]:
components.dtype, values.dtype, offsets.dtype

(dtype('int32'), dtype('float32'), dtype('int64'))

In [6]:
components.shape, values.shape, offsets.shape

((360757,), (360757,), (10001,))

In [7]:
# Ensure correct dtypes
components_correct = components.astype(np.int32)
values_correct = values.astype(np.float32)
offsets_correct = offsets.astype(np.int64)

print(f"Converted dtypes:")
print(f"  components: {components.dtype} -> {components_correct.dtype}")
print(f"  values: {values.dtype} -> {values_correct.dtype}")
print(f"  offsets: {offsets.dtype} -> {offsets_correct.dtype}")


Converted dtypes:
  components: int32 -> int32
  values: float32 -> float32
  offsets: int64 -> int64


In [ ]:
# Results from flat index search
distances_gt, groundtruth = flat_index.search(query_components=components_correct, query_values=values_correct, offsets=offsets_correct, k=k)
print(f"Ground truth computed for {n_queries} queries, k={k}")

Ground truth computed for 10000 queries, k=10


## Build SparsePlainHNSW Index

In [9]:
# Build HNSW index with n. neighbors m=32 and construction precision ef_construction=200
start = time.time()
hnsw = SparsePlainHNSW.build_from_file(data_path, metric="dotproduct", m=32, ef_construction=200)
end = time.time()
print(f"SparsePlainHNSW index built in {end - start} seconds")

SparsePlainHNSW index built in 28.658817529678345 seconds


## Search with Multiple Parameter Combinations

In [10]:
# Parameter configurations to test
configs = [
    {"ef_search": 20, "early_exit_threshold": None},
    {"ef_search": 100, "early_exit_threshold": None},
    {"ef_search": 15, "early_exit_threshold": 0.1}
]

results = []

for config in configs:
    ef_search = config["ef_search"]
    early_exit = config["early_exit_threshold"]
    
    # Search from file
    start = time.time()
    distances, results_ids = hnsw.search(components, values, offsets, k, ef_search=ef_search, early_exit_threshold=early_exit)
    elapsed = time.time() - start
    
    # Compute accuracy
    matches = 0
    for i in range(n_queries):
        gt_set = set(groundtruth[i*k:(i+1)*k])  # Ground truth for this query
        result_set = set(results_ids[i*k:(i+1)*k])  # Result for this query
        matches += len(gt_set & result_set)
    accuracy = (matches / (n_queries * k)) * 100
    avg_time = (elapsed / n_queries) * 10**6  # us per query
    
    results.append({
        "ef_search": ef_search,
        "early_exit_threshold": early_exit,
        "avg_time_ms": avg_time,
        "accuracy": accuracy
    })
    
    print(f"ef_search={ef_search}, early_exit={early_exit}: "
          f"avg_time={avg_time:.3f}us, accuracy={accuracy:.1f}%")

print("\n" + "="*60)
for r in results:
    print(f"Config: ef_search={r['ef_search']}, early_exit={r['early_exit_threshold']}")
    print(f"  Avg Query Time: {r['avg_time_ms']:.3f} us")
    print(f"  Accuracy: {r['accuracy']:.1f}%")
    print()

ef_search=20, early_exit=None: avg_time=85.961us, accuracy=91.9%
ef_search=100, early_exit=None: avg_time=276.310us, accuracy=96.4%
ef_search=15, early_exit=0.1: avg_time=128.249us, accuracy=94.6%

Config: ef_search=20, early_exit=None
  Avg Query Time: 85.961 us
  Accuracy: 91.9%

Config: ef_search=100, early_exit=None
  Avg Query Time: 276.310 us
  Accuracy: 96.4%

Config: ef_search=15, early_exit=0.1
  Avg Query Time: 128.249 us
  Accuracy: 94.6%

